# Control de un player de vídeo a través de gestos

## Paquetes necesarios

In [2]:
import cv2
import mediapipe as mp

![Hand landmarks.]("C:/Users/34611/OneDrive/Escritorio/GTDM/Tercero/Imagen/Proyecto/Control-de-player-via-gestos/images/hand_landmarks.png")

In [3]:
def get_finger_states(hand_landmarks):
    fingers = []

    # Pulgar (compara X en lugar de Y porque el pulgar se mueve lateralmente)
    if hand_landmarks.landmark[4].x > hand_landmarks.landmark[3].x:
        fingers.append(1)  # Extendido
    else:
        fingers.append(0)  # Doblado

    # Resto de los dedos: Índice, Medio, Anular, Meñique
    tips = [8, 12, 16, 20]
    pips = [6, 10, 14, 18]

    for tip, pip in zip(tips, pips):
        if hand_landmarks.landmark[tip].y < hand_landmarks.landmark[pip].y:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers  # [pulgar, índice, medio, anular, meñique]

In [4]:
# Detecta gesto: Mano abierta o puño
def detect_custom_gesture(fingers):
    # fingers = [pulgar, índice, medio, anular, meñique]
    #if fingers == [1, 1, 0, 0, 0]:
     #   return "L"
    #elif fingers == [0, 1, 1, 0, 0]:
      #  return "V"
    if fingers == [0, 1, 0, 0, 1]:
        return "Rock"
    #elif fingers == [0, 0, 0, 0, 0]:
     #   return "Puño"
    if fingers == [1, 1, 1, 1, 1]:
        return "Palma"
    else:
        return "Unknown"

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

# Configurar Selenium (modo visible o headless)
options = Options()
# options.add_argument('--headless')  # descomenta si quieres que sea invisible
options.add_argument("--autoplay-policy=no-user-gesture-required")

#WebDriverWait(driver, 10).until(
#    lambda d: d.execute_script("return video.readyState;") == 4
#)

In [ ]:
import pyautogui

In [ ]:
driver = webdriver.Chrome(options=options)
driver.get("https://areyousure.wavedash.com")


In [ ]:
def control(modo,pos_anterior,punta_indice,mov):
    global driver
    global previousMilis
    if modo == 'navegación':
        screen_w, screen_h = pyautogui.size()  #Dimensiones de la pantalla en píxeles
        if pos_anterior != None:
            if punta_indice.y-pos_anterior[8].y > 0.1:
                pyautogui.click()
                from selenium.webdriver.support.ui import WebDriverWait
                actual_url = driver.current_url 
                print(actual_url)
                if actual_url != 'areyousure.wavedash.com':
                    return 'player'
            else:  #Para que no haga el click y el movimiento a la vez
                pos_X = int(punta_indice.x*screen_w)
                pos_Y = int(punta_indice.y*screen_h)
                pyautogui.moveTo(pos_X, pos_Y)
        return 'navegación'
    else:
        currentMilis = time.time()
        if pos_anterior != None and currentMilis-previousMilis>0.5:
            if punta_indice.x-pos_anterior[8].x > 0.04:
                print('adelantar')
                mov = 'der-izq'
                driver.execute_script("video.currentTime += 5;")
            elif punta_indice.x-pos_anterior[8].x < -0.04:
                print('atrasar')
                mov = 'izq-der'
                driver.execute_script("video.currentTime -= 5;")
            if punta_indice.y-pos_anterior[8].y < -0.1:
                print('subir volumen')
                mov = 'abajo-arriba'
                driver.execute_script("if (video.volume <= 0.9){video.volume += 0.1}else {video.volume=1};")
            elif punta_indice.y-pos_anterior[8].y > 0.1:
                print('bajar volumen')
                mov = 'arriba-abajo'
                driver.execute_script("if (video.volume >= 0.1){video.volume -= 0.1}else {video.volume=0};")
            if mov == 'Palma':
                print('parar')
                driver.execute_script("video.pause();")
            elif mov == 'Rock':
                print('continuar')
                driver.execute_script("video.play();")
                
            if mov != 'Unknown':
                print('indice y:',punta_indice.y)
                print('movimiento punta_indice x', punta_indice.x-pos_anterior[8].x ,punta_indice.y-pos_anterior[8].y)
                previousMilis = currentMilis
        cv2.putText(image, f'Gesture: {mov}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        return 'player'

In [ ]:
import time
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
cap = cv2.VideoCapture(1)  #CAMBIAR A 0 SI NO VA
pos_anterior = None
modo = 'navegación'
currentMilis = 0
previousMilis = 0
with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print('No se puede leer la cámara')
            break

        try:
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = hands.process(image)
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    punta_indice = hand_landmarks.landmark[8]
                    dedos = get_finger_states(hand_landmarks)
                    mov = detect_custom_gesture(dedos) # Detecta el gesto de la palma 
                    modo = control(modo,pos_anterior,punta_indice,mov)
                    pos_anterior = hand_landmarks.landmark
                    
            cv2.imshow('Hand Gesture Recognition', image)
        except Exception as e:
            import traceback
            traceback.print_exc()
            
        
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [7]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)
mp_drawing = mp.solutions.drawing_utils

In [19]:
import math

def distancia_euclidiana(l1, l2):
    return math.sqrt((l1[0] - l2[0])**2 + (l1[1] - l2[1])**2 + (l1[2] - l2[2])**2)

def comparar_poses(pose_actual, pose_referencia, umbral=0.4):
    total_dist = 0
    num_puntos = len(pose_actual)

    for i in range(num_puntos):
        dist = distancia_euclidiana(pose_actual[i], pose_referencia[i])
        total_dist += dist

    promedio = total_dist / num_puntos
    #print(f"Distancia promedio: {promedio:.4f}")
    return promedio < umbral

In [20]:
def centrar_landmarks(landmarks):
    cadera_izq = landmarks[23]
    cadera_der = landmarks[24]
    x = (cadera_izq.x + cadera_der.x) / 2
    y = (cadera_izq.y + cadera_der.y) / 2
    z = (cadera_izq.z + cadera_der.z) / 2
    landmarks_centrados = []
    for l in landmarks:
        l_centrado = [0,0,0]
        l_centrado[0] = l.x - x
        l_centrado[1] = l.y - y
        l_centrado[2] = l.z - z
        landmarks_centrados.append(l_centrado)
    return landmarks_centrados

In [24]:
def mezclar(img,frame):
    img = img.astype(np.float32) / 255.0
    print(img.shape)
    frame = frame.astype(np.float32) / 255.0
    print(frame.shape)
    # Nivel de opacidad (alpha): 0 = totalmente transparente, 1 = totalmente opaca
    alpha = 0.5
    
    # Mezclar imagen con el fondo
    img_con_opacidad = cv2.addWeighted(frame, alpha, img, 1 - alpha, 0)
    
    # Volver a tipo uint8 para mostrar
    return (img_con_opacidad * 255).astype(np.uint8)

In [51]:
import numpy as np
img_bgr = cv2.imread("omniman_dummy.png")
sorpresa_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Procesar imagen
resultado = pose.process(sorpresa_rgb)

# Dibujar resultados si hay landmarks
if resultado.pose_landmarks:
    mp_drawing.draw_landmarks(img_bgr, resultado.pose_landmarks, mp_pose.POSE_CONNECTIONS)


    
        

    # Mostrar imagen con puntos dibujados

    cv2.imshow('win', img_bgr)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    sorpresa_landmarks = centrar_landmarks(resultado.pose_landmarks.landmark)
    img_bgr = cv2.imread("omniman.jpg")  #QUITAR PARA OTRAS
    omniman_bgr = cv2.resize(img_bgr,(640,480))
    print(sorpresa_bgr.shape)
else:
    print("No se detectó ninguna persona en la imagen.")

(480, 640, 3)


In [55]:
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)
cap = cv2.VideoCapture(1)
mp_drawing = mp.solutions.drawing_utils
if not cap.isOpened():
    print("Error al abrir la cámara")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convertir a RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Procesar con MediaPipe Pose
    result = pose.process(frame_rgb)

    # Dibujar resultados
    try:
        if result.pose_landmarks:
            landmarks_centrados = centrar_landmarks(result.pose_landmarks.landmark)
            if comparar_poses(landmarks_centrados, sorpresa_landmarks,0.4):
                #cv2.putText(frame, f'Pose: {'Sorpresa'}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                mezcla = mezclar(omniman_bgr,frame)
                cv2.imshow('Pose en Vivo', mezcla)
            else:
                #mp_drawing.draw_landmarks(frame, result.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                cv2.imshow('Pose en Vivo', frame)
    except Exception as e:
            import traceback
            traceback.print_exc()

    # Salir con 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar recursos
cap.release()
cv2.destroyAllWindows()

In [ ]:
print(type(resultado.pose_landmarks.landmark))